In [15]:
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

class Layer_Dense():
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.10 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    def forward(self, inputs):
        self.output = np.dot(inputs, self.weights) + self.biases

class Activation_ReLU():
    def forward(self, inputs):
        self.output = np.maximum(0, inputs)

class Activation_Softmax(): #cmn kepake di output terakhir
    def forward(self, inputs):
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        prob_values = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = prob_values

class Loss():
    def calculate(self, output, y): #output is output from model; y is intended model
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss

class Loss_CategoricalCrossentropy(Loss):
    def forward(self, y_pred, y_true):
        samples = len(y_pred)
        #jawaban benernya yg mana aja ; bisa 1 hot encoded atau ga b aja
        y_pred_clipped = np.clip(y_pred, 1e-7, 1-1e-7) #deket bgt ke 0 dan deket bgt ke 1

        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[range(samples), y_true]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(y_pred_clipped * y_true, axis=1)

        negative_log_likelihood = -np.log(correct_confidences)
        return negative_log_likelihood
        

X, y = spiral_data(samples=100,classes=3)

dense1 = Layer_Dense(2,3) #layer1
activation1 = Activation_ReLU()
dense2 = Layer_Dense(3, 3) #layer2
activation2 = Activation_Softmax()

dense1.forward(X)
activation1.forward(dense1.output)

dense2.forward(activation1.output)
activation2.forward(dense2.output)

print(activation2.output[:5])

loss_function = Loss_CategoricalCrossentropy()
loss = loss_function.calculate(activation2.output, y)
print("loss:",loss)

[[0.33333334 0.33333334 0.33333334]
 [0.33331734 0.33331832 0.33336434]
 [0.3332888  0.33329153 0.33341965]
 [0.33325943 0.33326396 0.33347666]
 [0.33323312 0.33323926 0.33352762]]
loss: 1.098445


In [ ]:
import torch
import torch.nn as nn
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()
torch.manual_seed(0)

X_np, y_np = spiral_data(samples=100, classes=3)
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.long)   # class indices, not one-hot

model = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 3),
)

with torch.no_grad():
    for layer in model:
        if isinstance(layer, nn.Linear):
            layer.weight.normal_(0.0, 1.0).mul_(0.10)
            layer.bias.zero_()

loss_function = nn.CrossEntropyLoss()

logits = model(X)
loss = loss_function(logits, y)

probs = torch.softmax(logits, dim=1)
print(probs[:5])
print("loss:", loss.item())

loss.backward()
print("dL/dW1:\n", model[0].weight.grad)


tensor([[0.3333, 0.3333, 0.3333],
        [0.3333, 0.3333, 0.3334],
        [0.3332, 0.3333, 0.3334],
        [0.3331, 0.3333, 0.3335],
        [0.3331, 0.3333, 0.3336]], grad_fn=<SliceBackward0>)
loss: 1.0986703634262085
dL/dW1:
 tensor([[ 0.0006, -0.0003],
        [-0.0009, -0.0015],
        [ 0.0022,  0.0009]])
